# Ingesta pipeline — end-to-end through the API

Exercises the whole T17 pipeline exactly as a real client would: upload a document
over HTTP, watch the SSE event stream, check the review queue, and record a human
decision. No shortcuts through the coordinator or the nodes directly — every call
below goes through `TestClient` and the same FastAPI routes a frontend would call.

Uses the **real** production `Container` (real SQL repos against `Settings.DATABASE_URL` — `data/classiflow.db` by default, real MarkItDown/OCR extraction, real embeddings) — unlike this notebook's earlier all-`TestContainer` version, this run's data is genuinely inspectable afterward in the real dev database. Only node3's SLM legitimacy call is mocked (`set_legitimacy()` below), so accept/review routing stays controllable/deterministic for the demo narrative regardless of what the real model would have said.

> **Heads up**: because this writes to the real, persistent `data/classiflow.db`, re-running this notebook will see `convenio_2_2013.pdf` (the "accepted" demo file) as an exact SHA-256 duplicate on the second and later runs, and node4 will correctly reject it instead of accepting it — that's real duplicate-detection behavior working as designed, not a bug. The checks below account for this (they check the *invariant*, not a fixed outcome).

## 1 — App setup: the real Container, JWT auth

In [1]:
from pathlib import Path

from fastapi.testclient import TestClient
from sqlalchemy import select
from sqlalchemy.ext.asyncio import async_sessionmaker, create_async_engine

import classiflow
from classiflow.api.app import create_app
from classiflow.database.base import Base
from classiflow.database.models import AllowedUser, Job
from classiflow.injections.production import Container
from classiflow.services.auth import encode_token
from classiflow.settings import Settings

Settings.JWT_SECRET_KEY = "playground-secret-key-not-for-prod-use-only-demo"

# Settings.DATABASE_URL defaults to a *relative* path ("./data/classiflow.db"), which
# breaks with "unable to open database file" when the kernel's cwd isn't the repo root
# (some IDE Jupyter integrations launch the kernel from the notebook's own directory).
# Anchor it to the actual package location instead -- overriding Settings.DATABASE_URL
# itself, not just our own engine below, since the FastAPI app's own db_session
# resolves through the same Settings value once a request comes in.
_project_root = Path(classiflow.__file__).parents[2]
_db_path = _project_root / "data" / "classiflow.db"
Settings.DATABASE_URL = f"sqlite+aiosqlite:///{_db_path.as_posix()}"

container = Container()
container.wire(packages=["classiflow"])

engine = create_async_engine(Settings.DATABASE_URL, echo=False)
session_factory = async_sessionmaker(engine, expire_on_commit=False)


async def _create_tables() -> None:
    # Idempotent -- only creates tables that don't already exist, so this is safe
    # to run against the already-migrated data/classiflow.db.
    async with engine.begin() as conn:
        await conn.run_sync(Base.metadata.create_all)


async def _seed_user(email: str) -> None:
    async with session_factory() as session:
        existing = await session.execute(select(AllowedUser).where(AllowedUser.email == email))
        if existing.scalar_one_or_none() is None:
            session.add(AllowedUser(email=email, is_active=True, is_blocked=False))
            await session.commit()


await _create_tables()
_EMAIL = "leonardo.heis@gmail.com"
await _seed_user(_EMAIL)

client = TestClient(create_app())
auth_headers = {"Authorization": f"Bearer {encode_token(_EMAIL)}"}

print(f"logged in as {_EMAIL}")
print(f"writing to {_db_path}")

c:\Users\leona\source\repos\Trabajo-Integrador\.venv\lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa
c:\Users\leona\source\repos\Trabajo-Integrador\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


logged in as leonardo.heis@gmail.com
writing to C:\Users\leona\source\repos\Trabajo-Integrador\data\classiflow.db


## 2 — Real sample PDFs and the SLM mock

Two actual municipal documents from `playground/samples/`, not synthetic bytes — real
file size, real magic-byte MIME sniffing, real SHA-256. Node 3's legitimacy check
calls a real LLM in production; here we swap it for `MockLlm` so the demo's
accept/review outcome is controllable without depending on what the real model
would decide. `set_legitimacy(...)` toggles what the mocked SLM decides.

> Extraction itself is **not** mocked here (unlike this notebook's earlier
> `TestContainer`-based version) — MarkItDown/OCR really run against these PDFs,
> exactly as production does. See `playground/stage1/text_extraction.ipynb` if you
> want to exercise extraction on its own, in isolation.

In [2]:
from pathlib import Path

import classiflow
import classiflow.ingesta.nodes.node3_content_validation as node3_module
from classiflow.ingesta.llm_provider import MockLlm

_SAMPLES_DIR = Path(classiflow.__file__).parent / "playground" / "samples"
_ACCEPTED_PDF = (_SAMPLES_DIR / "convenio_2_2013.pdf").read_bytes()
_REVIEW_PDF = (_SAMPLES_DIR / "ordenanza_6731_1999.pdf").read_bytes()

_SLM_LEGITIMATE = '{"is_legitimate": true, "confidence": 0.92, "reasoning": "official doc"}'
_SLM_NOT_LEGITIMATE = '{"is_legitimate": false, "confidence": 0.88, "reasoning": "looks like spam"}'


def set_legitimacy(*, is_legitimate: bool) -> None:
    response = _SLM_LEGITIMATE if is_legitimate else _SLM_NOT_LEGITIMATE
    node3_module.get_llm_langchain = lambda _path: MockLlm(response=response)


def upload(filename: str, file_bytes: bytes) -> dict[str, tuple[str, bytes, str]]:
    return {"file": (filename, file_bytes, "application/pdf")}


print(f"accepted-demo PDF: {len(_ACCEPTED_PDF):,} bytes")
print(f"review-demo PDF  : {len(_REVIEW_PDF):,} bytes")

accepted-demo PDF: 121,098 bytes
review-demo PDF  : 1,021,464 bytes


## 3 — Happy path: ingest a legitimate document

`POST /pipeline/ingest` returns `202` + a `job_id` immediately. `TestClient` runs
FastAPI's background tasks synchronously as part of the call, so by the time this
returns, the coordinator has already run node1 -> node2 -> node3 -> node4 to
completion -- in a real deployment this would happen after the response, which is
what `GET /{job_id}/events` (next section) is for: watching it happen live instead of
after the fact.

In [3]:
set_legitimacy(is_legitimate=True)

response = client.post(
    "/pipeline/ingest", files=upload("convenio_2_2013.pdf", _ACCEPTED_PDF), headers=auth_headers
)
print(f"status: {response.status_code}")
print(f"body  : {response.json()}")

accepted_job_id = response.json()["jobId"]

2026-08-13 12:37:33.138 | INFO     | classiflow.services.audit.service:record:37 - audit | job=cf755959-a45b-43f2-b596-47dc497938df node=node1_file_reception event=passed
2026-08-13 12:37:33.150 | INFO     | classiflow.services.audit.service:record:37 - audit | job=cf755959-a45b-43f2-b596-47dc497938df node=node2_format_validation event=passed
2026-08-13 12:37:33.895 | INFO     | classiflow.services.audit.service:record:37 - audit | job=cf755959-a45b-43f2-b596-47dc497938df node=node3_content_validation event=passed
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10239.71it/s]
2026-08-13 12:37:38.460 | INFO     | classiflow.services.audit.service:record:37 - audit | job=cf755959-a45b-43f2-b596-47dc497938df node=node4_duplicate_control event=passed


status: 202
body  : {'jobId': 'cf755959-a45b-43f2-b596-47dc497938df'}


## 4 — Watch the SSE event stream

Streams `node_update` events as they were emitted — `started` then `passed`/`failed`
per node — ending with a `pipeline` node event carrying `status: done`, at which
point the stream closes.

In [4]:
response = client.get(f"/pipeline/{accepted_job_id}/events", headers=auth_headers)
print(f"status: {response.status_code}\n")

for raw_block in response.text.split("event: node_update"):
    stripped = raw_block.strip()
    if stripped:
        print(stripped.removeprefix("data: "))

status: 200

{"job_id":"cf755959-a45b-43f2-b596-47dc497938df","node":"node1_file_reception","status":"started","timestamp":"2026-08-13T15:37:33.138606Z","detail":{}}
{"job_id":"cf755959-a45b-43f2-b596-47dc497938df","node":"node1_file_reception","status":"passed","timestamp":"2026-08-13T15:37:33.138606Z","detail":{}}
{"job_id":"cf755959-a45b-43f2-b596-47dc497938df","node":"node2_format_validation","status":"started","timestamp":"2026-08-13T15:37:33.148734Z","detail":{}}
{"job_id":"cf755959-a45b-43f2-b596-47dc497938df","node":"node2_format_validation","status":"passed","timestamp":"2026-08-13T15:37:33.148734Z","detail":{}}
{"job_id":"cf755959-a45b-43f2-b596-47dc497938df","node":"node3_content_validation","status":"started","timestamp":"2026-08-13T15:37:33.601229Z","detail":{}}
{"job_id":"cf755959-a45b-43f2-b596-47dc497938df","node":"node3_content_validation","status":"passed","timestamp":"2026-08-13T15:37:33.895909Z","detail":{}}
{"job_id":"cf755959-a45b-43f2-b596-47dc497938df","node":"n

## 5 — Review queue is empty

The document was accepted, so it never shows up in `GET /pipeline/review-queue`.

In [5]:
queue = client.get("/pipeline/review-queue", headers=auth_headers).json()
job_ids_in_queue = [item["jobId"] for item in queue]

print(f"jobs currently in review: {len(queue)}")
assert accepted_job_id not in job_ids_in_queue
print("accepted job correctly absent from the review queue")

jobs currently in review: 0
accepted job correctly absent from the review queue


## 6 — A document the SLM flags for review

Same upload, but this time the mocked SLM says the content isn't legitimate. Node 3
sets `needs_agent_review=True`, the coordinator routes to `review` instead of
`accepted`/`rejected`, and the job lands in the review queue with its full
`document_steps` history attached.

In [6]:
set_legitimacy(is_legitimate=False)

response = client.post(
    "/pipeline/ingest",
    files=upload("ordenanza_6731_1999.pdf", _REVIEW_PDF),
    headers=auth_headers,
)
review_job_id = response.json()["jobId"]
print(f"ingested job_id: {review_job_id}")

queue = client.get("/pipeline/review-queue", headers=auth_headers).json()
item = next(i for i in queue if i["jobId"] == review_job_id)

print(f"\nstatus          : {item['status']}")
print(f"filename        : {item['filename']}")
print(f"rejection_reason: {item['rejectionReason']}")
print("\ndocument_steps:")
for step in item["documentSteps"]:
    print(f"  [{step['stepOrder']}] {step['node']:<28} status={step['status']}")

2026-08-13 12:37:38.599 | INFO     | classiflow.services.audit.service:record:37 - audit | job=2ad8ea88-d74d-46ff-9763-d1960d8eb01d node=node1_file_reception event=passed
2026-08-13 12:37:38.609 | INFO     | classiflow.services.audit.service:record:37 - audit | job=2ad8ea88-d74d-46ff-9763-d1960d8eb01d node=node2_format_validation event=passed
2026-08-13 12:37:39.462 | INFO     | classiflow.services.audit.service:record:37 - audit | job=2ad8ea88-d74d-46ff-9763-d1960d8eb01d node=node3_content_validation event=failed


ingested job_id: 2ad8ea88-d74d-46ff-9763-d1960d8eb01d

status          : review
filename        : ordenanza_6731_1999.pdf
rejection_reason: SLM: looks like spam

document_steps:
  [1] node1_file_reception         status=passed
  [2] node2_format_validation      status=passed
  [3] node3_content_validation     status=failed


## 6b — extracted_text is persisted only for the non-accepted job

`PipelineService._finalize_job` stores the coordinator's extracted text on
`Job.extracted_text`, but only when the outcome isn't `accepted`. Checking the
*invariant* (accepted → `None`, anything else → populated) rather than a fixed
job-by-job expectation, since which specific outcome each demo job lands on can
vary across reruns against this real, persistent database (see the duplicate-hash
note in section 1).

In [7]:
# db_session is a dependency_injector Resource -- nothing in this codebase ever calls
# Closing[...]/shutdown_resources(), so the session opened for these requests has been
# flushed but never committed. shutdown_resources() runs get_session()'s post-yield
# `await session.commit()`, making the writes visible to session_factory's own,
# separate connection below.
await container.shutdown_resources()


async def _find_job(job_id: str) -> Job | None:
    async with session_factory() as session:
        result = await session.execute(select(Job).where(Job.job_id == job_id))
        return result.scalar_one_or_none()


for label, job_id in [("accepted-demo", accepted_job_id), ("review-demo", review_job_id)]:
    job = await _find_job(job_id)
    assert job is not None
    print(f"{label}: status={job.status!r} extracted_text={job.extracted_text!r}")
    if job.status == "accepted":
        assert job.extracted_text is None
    else:
        assert job.extracted_text is not None

accepted-demo: status='accepted' extracted_text=None
review-demo: status='review' extracted_text='..\n\n, r. r::1. .  r  C. ulll •\n\nr t:. u c\n\n~1rg- --~\n\nMUtll CIPAillU  t E ~tlS A l.\nR E G  I S T  ~ A_ O _Q_\n\n2.2 FE B .19 g l~\nDlrec.  Me1a  8r1L dt  htrnocl\nf Ar(hlvo  OoDQtal\n\nLA MUNICIPALIDAD  DE  ROSARIO HA SANCIONADO LA SIGUIENTE\n\nORDENANZA\n\n(Nº 6.731)\n\nHonorable Concejo:\n\nLa  Comisión  de  Planeamiento  y  Urbanismo  ha  considerado  el  Mensaje\n79/98  SPI  enviado  por  el  Departamento  Ejecutivo  relacionado  con  proyecto  de  Ordenanza  me\ndiante el  cual  se aprueba el anteproyecto de urbanización referido a la Primera Fase del  Centro de\nRenovación Urbana Scalabrini Ortiz.\n\nVisto el acta labrada por la Comisión Técnica de Urbanización que informa\nsobre  el  cumplimiento, en  función  del  nivel  de  presentación,  de  los  parámetros  reglamentarios\nestablecidos para esta Primera Fase.\n\nQue  el  esquema  vehicular  del  área,  estructurado  no 

## 7 — Record a human decision

A reviewer accepts the flagged document anyway. `POST /pipeline/{job_id}/decision`
records who decided and why, updates the job's status, and the job then disappears
from the review queue.

In [8]:
response = client.post(
    f"/pipeline/{review_job_id}/decision",
    json={"decision": "accept", "notes": "Verified manually, looks legitimate"},
    headers=auth_headers,
)
print(f"status: {response.status_code}")

queue = client.get("/pipeline/review-queue", headers=auth_headers).json()
print(f"still in review queue: {review_job_id in [i['jobId'] for i in queue]}")

status: 200
still in review queue: False


## 8 — Guardrails: auth, unknown jobs, wrong state

Quick check that the error paths behave as designed:
- No token -> `401`
- Unknown `job_id` -> `404`
- Deciding on a job that isn't in `review` anymore -> `409`

In [9]:
no_auth = client.post("/pipeline/ingest", files=upload("x.pdf", _ACCEPTED_PDF))
print(f"no auth header       -> {no_auth.status_code}")

unknown = client.get("/pipeline/no-such-job/events", headers=auth_headers)
print(f"unknown job events    -> {unknown.status_code}")

already_decided = client.post(
    f"/pipeline/{review_job_id}/decision",
    json={"decision": "accept"},
    headers=auth_headers,
)
print(f"decide on non-review  -> {already_decided.status_code}")

no auth header       -> 401
unknown job events    -> 404
decide on non-review  -> 409
